# 02 · Base compartilhada (filtro de KPI)

**Projeto:** Cronos · Challenge FIAP 2026 com Locaweb
**Autor(es):** Ana Beatriz Costa de Oliveira · Hygor Abrantes · Igor Vignola
**Data de criação:** 21/07/2026
**Última atualização:** 28/07/2026

## Objetivo

Produzir a base que todos os modelos reutilizam: carregar o dataset, corrigir tipos,
aplicar o filtro oficial de elegibilidade ao KPI e salvar em
`data/interim/incidentes_kpi.parquet`, com uma linha por incidente elegível.
Cada modelo remodela essa base conforme sua necessidade (o Prophet agrega por dia; o modelo
de risco usa uma linha por incidente; o índice de saúde agrega por produto).

## Dependências

- Python 3.11+
- pandas, openpyxl (leitura do Excel), pyarrow (escrita do Parquet)

## Entradas

- `assets/Materal LocalWeb/LW-DATASET.xlsx` (aba `Dataset Geral`)

## Saídas

- `data/interim/incidentes_kpi.parquet`

## 1. Setup

In [1]:
# Stdlib
from pathlib import Path

# Third-party
import pandas as pd

pd.set_option('display.max_columns', None)

# Resolve caminhos (funciona rodando da raiz do repo ou de notebooks/)
_cands = [Path('assets/Materal LocalWeb/LW-DATASET.xlsx'),
          Path('../assets/Materal LocalWeb/LW-DATASET.xlsx')]
XLSX = next(p for p in _cands if p.exists())
REPO = XLSX.parents[2]
INTERIM = REPO / 'data' / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)
print('dataset:', XLSX)
print('saida:  ', INTERIM / 'incidentes_kpi.parquet')

dataset: ..\assets\Materal LocalWeb\LW-DATASET.xlsx
saida:   ..\data\interim\incidentes_kpi.parquet


## 2. Carga dos dados

Apenas leitura, sem transformação.

In [2]:
df_raw = pd.read_excel(XLSX, sheet_name='Dataset Geral')

# Invariante do dataset oficial da Locaweb; se mudar, todos os numeros a jusante mudam.
assert len(df_raw) == 122_543, f'Esperadas 122.543 linhas no dataset bruto, lidas {len(df_raw):,}'

print(f'Linhas: {len(df_raw):,} | Colunas: {df_raw.shape[1]}')
df_raw.head(3)

Linhas: 122,543 | Colunas: 19


,Número,Prioridade,Produto,Categoria,Subcategoria,Grupo designado,Item de configuração,Aberto,Resolvido,Encerrado,Duração,Código de fechamento,Descrição resumida,Solução,Aberto por,Incidente Pai,Status,Entrou para KPI?,KPI Violado?
0,INC8654273,3 - Média,NaN,NaN,NaN,Team14,IC00001,2025-12-31 23:45:18,NaT,2025-12-31 23:45:32,14,NaN,Problem: Apache Busy Workers,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
1,INC8654270,4 - Baixa,NaN,NaN,NaN,Team14,IC00002,2025-12-31 23:39:36,NaT,2025-12-31 23:43:05,209,NaN,Problem: Check Application Monitoring,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
2,INC8654264,4 - Baixa,NaN,NaN,NaN,Team14,NaN,2025-12-31 23:23:10,NaT,2025-12-31 23:25:00,110,NaN,Problem: Alarm Application Monitoring database...,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN


## 3. Tipagem

Converte datas para `datetime` e `Duração` (em segundos) para numérico. As demais colunas
não exigem tratamento adicional; a consistência foi verificada na análise exploratória
(notebook 01). Ao final, verifica-se quantos nulos a conversão com `errors='coerce'` gerou
em cada coluna. A coluna `Aberto` não pode ter nulos: ela alimenta toda a série temporal
usada nos modelos.

In [3]:
cols_convertidas = ['Aberto', 'Resolvido', 'Encerrado', 'Duração']
nulos_antes = df_raw[cols_convertidas].isna().sum()

for col in ['Aberto', 'Resolvido', 'Encerrado']:
    df_raw[col] = pd.to_datetime(df_raw[col], errors='coerce')
df_raw['Duração'] = pd.to_numeric(df_raw['Duração'], errors='coerce')

print(df_raw[cols_convertidas].dtypes)
print("\nNulos gerados pela conversão (errors='coerce'):")
print((df_raw[cols_convertidas].isna().sum() - nulos_antes).to_string())

# 'Aberto' e a coluna que alimenta toda a serie temporal a jusante.
assert df_raw['Aberto'].notna().all(), 'Coluna Aberto contém nulos após a conversão; a série temporal depende dela.'

Aberto       datetime64[ns]
Resolvido    datetime64[ns]
Encerrado    datetime64[ns]
Duração               int64
dtype: object

Nulos gerados pela conversão (errors='coerce'):
Aberto       0
Resolvido    0
Encerrado    0
Duração      0


## 4. Filtro de elegibilidade ao KPI

Utiliza-se o **campo oficial** `Entrou para KPI? == 'SIM'`, com aderência de 99,88% à regra
do dicionário (incidente pai vazio + prioridade 1/2/3 + status diferente de 'Sem Intervenção').
Opta-se pelo campo oficial em vez de reimplementar a regra, evitando divergências de
interpretação.

In [4]:
df_kpi = df_raw[df_raw['Entrou para KPI?'] == 'SIM'].copy()

# Invariante do filtro oficial; numero citado em docs e slides do projeto.
assert len(df_kpi) == 25_600, f'Esperados 25.600 incidentes elegíveis ao KPI, obtidos {len(df_kpi):,}'

print(f'Total:      {len(df_raw):,}')
print(f'Elegiveis:  {len(df_kpi):,}  ({len(df_kpi)/len(df_raw)*100:.0f}%)')
print(f'Periodo:    {df_kpi["Aberto"].min().date()} a {df_kpi["Aberto"].max().date()}')

Total:      122,543


Elegiveis:  25,600  (21%)
Periodo:    2023-01-02 a 2025-12-31


## 5. Colunas de calendário + concentração temporal

Colunas derivadas de `Aberto` para uso nos modelos. Confirma-se também a concentração
temporal: os registros elegíveis estão majoritariamente em **2025** (~1 ano de dado denso).

O recorte exploratório da série diária considera apenas P2 e P3. A prioridade P1 não aparece
porque tem um único registro em todo o dataset (1 em 122.543), o que não forma uma série.

In [5]:
df_kpi['dia'] = df_kpi['Aberto'].dt.normalize()
df_kpi['dia_semana'] = df_kpi['Aberto'].dt.dayofweek   # 0 = segunda
df_kpi['hora'] = df_kpi['Aberto'].dt.hour
df_kpi['ano'] = df_kpi['Aberto'].dt.year
df_kpi['mes'] = df_kpi['Aberto'].dt.month

print('Elegiveis por ano:')
print(df_kpi['ano'].value_counts().sort_index().to_string())

print('\nSerie diaria 2025 (o que o Prophet vai prever):')
df_2025 = df_kpi[df_kpi['ano'] == 2025]
# 'Prioridade' vem como texto no formato 'N - Nome' (ex.: '2 - Alta', '3 - Média');
# o prefixo numérico identifica a prioridade.
for tag, pref in [('P2', '2'), ('P3', '3')]:
    s = df_2025[df_2025['Prioridade'].astype(str).str.startswith(pref)].groupby('dia').size()
    print(f'  {tag}: media {s.mean():.1f}/dia | min {s.min()} | max {s.max()}')
print()
print('distribuição dos elegíveis por ano:')
_por_ano = df_kpi['ano'].value_counts().sort_index()
for _a, _n in _por_ano.items():
    print(f'  {_a}: {_n:,} ({_n/len(df_kpi)*100:.1f}%)'.replace(',', '.'))
print(f'  2023 e 2024 somam {_por_ano.reindex([2023, 2024]).sum():.0f} registros')


Elegiveis por ano:
ano
2023       87
2024      357
2025    25156

Serie diaria 2025 (o que o Prophet vai prever):
  P2: media 14.1/dia | min 3 | max 43
  P3: media 54.8/dia | min 1 | max 168

distribuição dos elegíveis por ano:
  2023: 87 (0.3%)
  2024: 357 (1.4%)
  2025: 25.156 (98.3%)
  2023 e 2024 somam 444 registros


## 6. Salvar a base

O Parquet relê rápido: os próximos notebooks começam com um `read_parquet` em vez de
reprocessar o Excel. O arquivo preserva as 19 colunas originais e adiciona as 5 colunas
de calendário (`dia`, `dia_semana`, `hora`, `ano`, `mes`), conforme o schema impresso abaixo.

In [6]:
print('Schema do parquet (19 colunas originais + 5 de calendário):')
print(df_kpi.dtypes.to_string())

out = INTERIM / 'incidentes_kpi.parquet'
df_kpi.to_parquet(out, index=False)
print(f'\nSalvo: {out}')
print(f'Shape: {df_kpi.shape[0]:,} linhas x {df_kpi.shape[1]} colunas')

Schema do parquet (19 colunas originais + 5 de calendário):
Número                          object
Prioridade                      object
Produto                         object
Categoria                       object
Subcategoria                    object
Grupo designado                 object
Item de configuração            object
Aberto                  datetime64[ns]
Resolvido               datetime64[ns]
Encerrado               datetime64[ns]
Duração                          int64
Código de fechamento            object
Descrição resumida              object
Solução                         object
Aberto por                      object
Incidente Pai                   object
Status                          object
Entrou para KPI?                object
KPI Violado?                    object
dia                     datetime64[ns]
dia_semana                       int32
hora                             int32
ano                              int32
mes                              int32



Salvo: ..\data\interim\incidentes_kpi.parquet
Shape: 25,600 linhas x 24 colunas


## 7. Conclusões e próximos passos

- Base pronta: **uma linha por incidente elegível ao KPI**, tipada, com colunas de calendário.
- Achado que guia a modelagem: **~98% dos elegíveis estão em 2025** → treinar em 2025,
  sazonalidade semanal ligada, anual desligada.
- Próximo notebook: `03_previsao_volume`, que agrega esta base por dia (P2 e P3 separados) e
  treina o Prophet.